In [1]:
# --- Colab bootstrap -------------------------------------------------------
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    !rm -rf /tmp/cbet6e
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------


# One chemical, one measurement, five compartments (Illustrations 12.1-3, 12.5-1 and 12.5-3)

Benzo[a]pyrene is a combustion product and a carcinogen. It is also the chapter's best
single thread: **one measured number carries it from a beaker to a river.**

The measurement is a solubility -- $x_{\rm BP} = 3.37\times10^{-10}$ in water at 25 °C.
From it:

1. **Illustration 12.1-3** turns that solubility into an activity coefficient. Because
   benzo[a]pyrene is a *solid*, $\gamma\ne1/x^{\rm sat}$: the equilibrium is with the
   pure solid, so the fugacity ratio of Eq. 12.1-6 has to be divided out first. The
   answer is $\gamma^\infty = 3.76\times10^8$, which SIS's own Comment calls out as a
   reminder that activity coefficients are not always small corrections.
2. **Illustration 12.5-1** turns that activity coefficient into an air-water partition
   coefficient through Eq. 12.5-8.
3. **Illustration 12.5-3** turns that partition coefficient, plus a measured
   $K_{\rm OW}$, into concentrations in air, soil, sediment and fish -- and compares them
   with field data spanning eight orders of magnitude.

**The whole model is one idea repeated.** Each compartment is treated as *the fraction of
itself that behaves like octanol*: lipid in fish, organic carbon in soil and sediment.
Equations 12.5-9 to 12.5-11 are $K_{\rm OW}$ times that fraction, with an empirical 0.4
for organic carbon which Sec. 12.5 flags as empirical.

⚠️ **One number did not carry cleanly between the two illustrations in the 5e**, and
tracking it down is worked out below. Illustration 12.5-1 computed
$K_{\rm AW} = 5.844\times10^{-5}$ while Illustration 12.5-3 quoted and used
$5.884\times10^{-5}$ -- and neither is what Eq. 12.5-8 returns. All three numbers are now
accounted for: the equation gives $5.813\times10^{-5}$; the original worksheet carried a
spurious factor of the total pressure, 1.013 bar, which gives $5.883\times10^{-5}$ and is
what Illustration 12.5-3 used; and $5.844$ is a digit transposition of that. **The 6e
prints the equation's own answer, $5.813\times10^{-5}$, in both places**, which moves
exactly one number in Illustration 12.5-3's results table -- the air concentration, from
1.66 to 1.64 ng/m$^3$, inside the reported range of 1.3 to 7.1.

SIS is Stanley I. Sandler, *Chemical, Biochemical, and Engineering Thermodynamics*.

Eric Furst
August 2026

In [2]:

import sys; sys.path.append("..")
import numpy as np

from thermo import sle
from thermo.partition import (air_water_partition, compartment_partition,
                              compartment_concentrations, COMPARTMENTS,
                              EQ_12_5_8_CONSTANT)

T = 298.15

# --- Illustration 12.1-3 ---------------------------------------------------
TM_BP    = 178.1 + 273.15          # melting point, K
DH_FUS   = 15100.0                 # J/mol
X_SAT    = 3.37e-10                # mole fraction in water at 25 C
P_VAP_PA = 2.13e-5                 # Pa, the extrapolated LIQUID vapor pressure
LOG_KOW  = 6.04                    # measured

print("  Illustration 12.1-3: an activity coefficient from a solubility")
ratio = float(np.exp(sle.ln_x_gamma(T, TM_BP, DH_FUS)))
gamma = float(sle.activity_coefficient(X_SAT, T, TM_BP, DH_FUS))
print(f"    f^S / f^L         = {ratio:.4f}")
print(f"    gamma^inf         = {gamma:.4g}      SIS 3.76e8")
print(f"    1/x_sat would be  = {1 / X_SAT:.4g}"
      f"   -- high by a factor of {1 / ratio:.2f}")
print(f"\n    ⭐ That factor is why a melting point appears in what looks like a")
print(f"    solubility calculation. Benzo[a]pyrene is a SOLID at 25 C, so a saturated")
print(f"    aqueous solution is in equilibrium with the pure solid, not with a pure")
print(f"    liquid -- and the ratio of those two fugacities is Eq. 12.1-6's exponential.")
print(f"    Take gamma = 1/x_sat and you are a factor of {1 / ratio:.1f} high.")

  Illustration 12.1-3: an activity coefficient from a solubility
    f^S / f^L         = 0.1266
    gamma^inf         = 3.757e+08      SIS 3.76e8
    1/x_sat would be  = 2.967e+09   -- high by a factor of 7.90

    ⭐ That factor is why a melting point appears in what looks like a
    solubility calculation. Benzo[a]pyrene is a SOLID at 25 C, so a saturated
    aqueous solution is in equilibrium with the pure solid, not with a pure
    liquid -- and the ratio of those two fugacities is Eq. 12.1-6's exponential.
    Take gamma = 1/x_sat and you are a factor of 7.9 high.



## Illustration 12.5-1, and the constant in Eq. 12.5-8

Equation 12.5-8 is $K_{\rm AW} = 0.2164\,\gamma^\infty P^{\rm vap}/T$ for $P^{\rm vap}$
in bar. The 0.2164 is not fitted -- it is three unit conversions multiplied together, and
worth unpacking because getting one of them wrong is exactly what appears to have happened
between the two illustrations:

$$\underbrace{1.218\times10^4}_{\text{Eq. 12.5-6, ideal gas}}
  \Big/ \underbrace{5.556\times10^4}_{\text{Eq. 12.5-7},\;10^6/18}
  \Big/ \underbrace{1.013}_{P\ \text{in Eq. 12.5-5}} = 0.2164$$

The molecular weight cancels between Eqs. 12.5-6 and 12.5-7, which is why it does not
appear. The atmospheric pressure comes in because Eq. 12.5-5 is
$y_iP = x_i\gamma_i^\infty P_i^{\rm vap}$, so $y_i/x_i$ carries a $1/P$.

In [3]:

print("  Where 0.2164 comes from")
print(f"    1.218e4 / 5.556e4        = {1.218e4 / 5.556e4:.5f}   SIS's own 0.2192")
print(f"    divided by P = 1.013 bar = {EQ_12_5_8_CONSTANT:.5f}   SIS's own 0.2164")

print("\n  Illustration 12.5-1, three ways of arriving at K_AW")
K_EQ = float(air_water_partition(3.76e8, P_VAP_PA * 1e-5, T))
K_NOP = float(air_water_partition(3.76e8, P_VAP_PA * 1e-5, T,
                                  constant=1.218e4 / 5.556e4))
K_EXACT = float(air_water_partition(gamma, P_VAP_PA * 1e-5, T))
print(f"    Eq. 12.5-8, gamma = 3.76e8 as printed   K_AW = {K_EQ:.4e}")
print(f"    the same with the unrounded gamma       K_AW = {K_EXACT:.4e}")
print(f"    the same but WITHOUT the 1/1.013        K_AW = {K_NOP:.4e}")
print(f"\n    SIS, Illustration 12.5-1               K_AW = 5.844e-05")
print(f"    SIS, Illustration 12.5-3 (quoted and used)   5.884e-05")
K_NOP_EXACT = float(air_water_partition(gamma, P_VAP_PA * 1e-5, T,
                                        constant=1.218e4 / 5.556e4))
print(f"    the same, unrounded gamma AND no 1/1.013 K_AW = {K_NOP_EXACT:.4e}")
print(f"\n  ⚠️ All three printed numbers are now accounted for, which is what makes this")
print(f"  a finding rather than a story:")
print(f"    * Eq. 12.5-8 as printed returns {K_EQ:.4e} -- and its constant 0.2164 is the")
print(f"      right one: V_W/R = 0.2167 with T in K and P^vap in bar.")
print(f"    * dropping the 1/1.013, with the UNROUNDED gamma = {gamma:.4e} from")
print(f"      Illustration 12.1-3, gives {K_NOP_EXACT:.4e} -- which is Illustration")
print(f"      12.5-3's printed 5.884e-5 to four figures. That is the original worksheet.")
print(f"    * 5.844e-5 in Illustration 12.5-1 is a digit transposition of 5.884, and is")
print(f"      not the output of any route.")
print(f"\n  The 6e prints {K_EQ:.4e} in both illustrations. Only the air concentration in")
print(f"  Illustration 12.5-3 moves, by {abs(K_EQ / 5.884e-5 - 1) * 100:.1f} %.")

  Where 0.2164 comes from
    1.218e4 / 5.556e4        = 0.21922   SIS's own 0.2192
    divided by P = 1.013 bar = 0.21641   SIS's own 0.2164

  Illustration 12.5-1, three ways of arriving at K_AW
    Eq. 12.5-8, gamma = 3.76e8 as printed   K_AW = 5.8131e-05
    the same with the unrounded gamma       K_AW = 5.8084e-05
    the same but WITHOUT the 1/1.013        K_AW = 5.8887e-05

    SIS, Illustration 12.5-1               K_AW = 5.844e-05
    SIS, Illustration 12.5-3 (quoted and used)   5.884e-05
    the same, unrounded gamma AND no 1/1.013 K_AW = 5.8839e-05

  ⚠️ All three printed numbers are now accounted for, which is what makes this
  a finding rather than a story:
    * Eq. 12.5-8 as printed returns 5.8131e-05 -- and its constant 0.2164 is the
      right one: V_W/R = 0.2167 with T in K and P^vap in bar.
    * dropping the 1/1.013, with the UNROUNDED gamma = 3.7570e+08 from
      Illustration 12.1-3, gives 5.8839e-05 -- which is Illustration
      12.5-3's printed 5.884e-5 to fou


## Illustration 12.5-3: five compartments, eight orders of magnitude

Given the concentration in water -- $2.82\times10^4$ ng/m$^3$ in southern Ontario -- every
other compartment follows from a partition coefficient. The arithmetic is easy; the **unit
bookkeeping is where this illustration is actually difficult**, because each compartment
answer is wanted per unit *volume* and the partition coefficients are per unit *mass*, and
the two differ by the compartment density: 1500 kg/m$^3$ for soil, 1420 for sediment, about
1000 for biota.

Getting that backwards is a factor of 1.5 for soil. `compartment_concentrations` returns
both, named, for exactly that reason.

In [4]:

C_WATER = 2.82e4                   # ng/m^3, reported for southern Ontario
KOW = 10.0 ** LOG_KOW
# The 6e prints Eq. 12.5-8's own answer in both illustrations. The 5e printed
# 5.844e-5 in Illustration 12.5-1 and used 5.884e-5 in Illustration 12.5-3, and
# neither is what the equation returns -- see the cell above and the note below.
K_AW = K_EQ                        # 5.813e-5, from Eq. 12.5-8 as printed
K_AW_5E = 5.884e-5                 # what Illustration 12.5-3 carried forward

print(f"  K_OW = 10^{LOG_KOW} = {KOW:.4g}    SIS 1.096e6")
print("\n  The partition coefficients, Eqs. 12.5-9 to 12.5-11")
SIS_K = {"biota": 5.48e4, "soil": 8768.0, "sediment": 21920.0}
print(f"    {'compartment':<11} {'w':>6} {'factor':>7} {'K':>12} {'SIS':>11}")
for name in ("biota", "soil", "sediment"):
    c = COMPARTMENTS[name]
    K = float(compartment_partition(KOW, name))
    print(f"    {name:<11} {c['w']:6.2f} {c['factor']:7.1f} {K:12.4g}"
          f" {SIS_K[name]:11.4g}")
print(f"    (SIS's biota value is 0.05 x K_OW and is not printed as a K, only used.)")

res = compartment_concentrations(C_WATER, KOW, K_AW)
res_5e = compartment_concentrations(C_WATER, KOW, K_AW_5E)
SIS_CALC = {"air": 1.66, "soil": 3.71e8, "sediment": 8.78e8, "biota": 1.55e9}
# Only AIR moves when K_AW changes: soil, sediment and biota come from K_OW
# through Eqs. 12.5-9 to 12.5-11 and never see the air-water coefficient. That is
# why the 6e correction to K_AW reaches exactly one number in the printed table.
SIS_6E = {"air": 1.64}
SIS_REPORTED = {"air": "1.3 to 7.1", "soil": "1.1e8",
                "sediment": "0.8e8 to 3e8", "biota": "1.4e8"}
print(f"\n  Concentrations, ng/m^3")
print(f"    {'compartment':<11} {'recomputed':>12} {'SIS calc':>11} {'ratio':>7}"
      f"   reported (Mackay and Paterson)")
print(f"    {'water':<11} {C_WATER:12.4g} {C_WATER:11.4g} {1.0:7.3f}"
      f"   2.82e4 (the input)")
for name in ("air", "soil", "sediment", "biota"):
    v = float(res[name]["per_volume"])
    print(f"    {name:<11} {v:12.4g} {SIS_CALC[name]:11.4g}"
          f" {v / SIS_CALC[name]:7.3f}   {SIS_REPORTED[name]}")

print(f"\n  What the K_AW correction moves, and what it does not")
for name in ("air", "soil", "sediment", "biota"):
    a = float(res[name]["per_volume"]); b = float(res_5e[name]["per_volume"])
    flag = "  <- the only one that moves" if abs(a / b - 1) > 1e-9 else ""
    print(f"    {name:<11} K_AW = 5.813e-5: {a:11.4g}   5.884e-5: {b:11.4g}"
          f"  {(a / b - 1) * 100:+5.2f} %{flag}")
print(f"    so the 6e prints {SIS_6E['air']} ng/m^3 for air where the 5e printed"
      f" {SIS_CALC['air']}, and the")
print(f"    reported range {SIS_REPORTED['air']} ng/m^3 contains both.")

print(f"\n  And the ppm-by-weight figures the illustration also prints")
# per_mass comes back in ng per GRAM of compartment, which is ppb by weight; ppm is
# that over 1000. Getting this step wrong is a clean factor of 1e3 that still lands on
# a plausible-looking number, which is why `compartment_concentrations` names the basis
# of both columns instead of returning one bare array.
for name, sis in (("soil", 0.247), ("sediment", 0.618), ("biota", 1.55)):
    pm = float(res[name]["per_mass"]) / 1e3     # ng/g = ppb -> ppm by weight
    print(f"    {name:<11} {pm:8.4f} ppm    SIS {sis}")

  K_OW = 10^6.04 = 1.096e+06    SIS 1.096e6

  The partition coefficients, Eqs. 12.5-9 to 12.5-11
    compartment      w  factor            K         SIS
    biota         0.05     1.0    5.482e+04    5.48e+04
    soil          0.02     0.4         8772        8768
    sediment      0.05     0.4    2.193e+04   2.192e+04
    (SIS's biota value is 0.05 x K_OW and is not printed as a K, only used.)

  Concentrations, ng/m^3
    compartment   recomputed    SIS calc   ratio   reported (Mackay and Paterson)
    water           2.82e+04    2.82e+04   1.000   2.82e4 (the input)
    air                1.639        1.66   0.988   1.3 to 7.1
    soil            3.71e+08    3.71e+08   1.000   1.1e8
    sediment       8.781e+08    8.78e+08   1.000   0.8e8 to 3e8
    biota          1.546e+09    1.55e+09   0.997   1.4e8

  What the K_AW correction moves, and what it does not
    air         K_AW = 5.813e-5:       1.639   5.884e-5:       1.659  -1.20 %  <- the only one that moves
    soil        K_AW 

In [5]:

print("  What the comparison with field data actually shows")
span = SIS_CALC["biota"] / SIS_CALC["air"]
print(f"    The calculation spreads one chemical over {np.log10(span):.1f} orders of")
print(f"    magnitude between air and fish, from {SIS_CALC['air']:.2g} to"
      f" {SIS_CALC['biota']:.2g} ng/m^3.")
print(f"    Reported values span the same range. That is the result: a model with no")
print(f"    fitted parameter beyond the empirical 0.4 puts every compartment within")
print(f"    about an order of magnitude of measurement.")
print(f"\n    But look at WHERE it fails, because the failures are not random:")
for name in ("air", "soil", "sediment", "biota"):
    rep = {"air": 3.0, "soil": 1.1e8, "sediment": 1.5e8, "biota": 1.4e8}[name]
    calc = float(res[name]["per_volume"])
    print(f"      {name:<9} calculated / reported = {calc / rep:6.2f}")
print(f"    Air is close. Soil and sediment are high by 3 to 6. Fish is high by 11.")
print(f"    The trend is monotone in how much the model had to ASSUME: air needs only")
print(f"    a vapor pressure, soil and sediment need the empirical 0.4 AND an organic")
print(f"    fraction, and biota needs the assumption that lipid IS octanol. The")
print(f"    disagreement grows with the number of stand-ins, which is what you would")
print(f"    hope for -- it means the thermodynamics is not what is wrong.")

  What the comparison with field data actually shows
    The calculation spreads one chemical over 9.0 orders of
    magnitude between air and fish, from 1.7 to 1.6e+09 ng/m^3.
    Reported values span the same range. That is the result: a model with no
    fitted parameter beyond the empirical 0.4 puts every compartment within
    about an order of magnitude of measurement.

    But look at WHERE it fails, because the failures are not random:
      air       calculated / reported =   0.55
      soil      calculated / reported =   3.37
      sediment  calculated / reported =   5.85
      biota     calculated / reported =  11.04
    Air is close. Soil and sediment are high by 3 to 6. Fish is high by 11.
    The trend is monotone in how much the model had to ASSUME: air needs only
    a vapor pressure, soil and sediment need the empirical 0.4 AND an organic
    fraction, and biota needs the assumption that lipid IS octanol. The
    disagreement grows with the number of stand-ins, which i


## Your turn

1. Section 12.5 takes the organic-carbon-water partition coefficient as 40 % of
   $K_{\rm OW}$ and says so is "empirically found." Fit that fraction instead, from the
   reported soil and sediment concentrations. Does one number serve both, and how far is
   it from 0.4?
2. The biota answer is the furthest from measurement, and it rests on lipid being
   octanol at a weight fraction of 0.05. What lipid fraction would reproduce the reported
   $1.4\times10^8$ ng/m$^3$? Is that a plausible fish?
3. Problem 12.5-2 gives four insecticides with their water solubilities and
   $\log_{10}K_{\rm OW}$, and asks for the concentration in a fish in a saturated tank.
   Do all four, and notice that solubility and $K_{\rm OW}$ run in opposite directions --
   which of the two decides the answer?
4. Problem 12.5-3 is a closed terrarium: 10 m$^3$ total, contaminated with 10 mg of
   benzene. That is a *mass balance* problem, not a partition-coefficient problem -- the
   total is fixed and shared out. Set it up and solve for all four compartments, then do
   it again for DDT and explain the difference in one sentence.
5. Illustration 12.1-3's route needs a melting point and a heat of fusion because the
   solute is a solid. What would change if benzo[a]pyrene were a liquid at 25 °C, and
   which of this notebook's numbers would move?
6. The vapor pressure used in Illustration 12.5-1 is described as *extrapolated* -- it is
   the pressure of a liquid that does not exist at 25 °C, the same hypothetical phase as
   in Illustration 12.1-1. Trace how a 20 % error in that extrapolation propagates to the
   air concentration, and compare that sensitivity with the 0.4.